In [0]:
# Préparation
from pyspark.sql import functions as F

PRICING = "alpenia_dev.bronze.brz_pricing"
EVENTS  = "alpenia_dev.bronze.brz_web_events"

def history(table, n=10):
    display(
        spark.sql(f"DESCRIBE HISTORY {table}")
        .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
        .orderBy(F.desc("version"))
        .limit(n)
    )

def last_version(table):
    return spark.sql(f"DESCRIBE HISTORY {table}").agg(F.max("version")).first()[0]

# Time travel et RESTORE sur brz_pricing

In [0]:
# Avant
v_before = last_version(PRICING)
print("Dernière version correcte :", v_before)
display(spark.table(PRICING).orderBy("city", "product"))

In [0]:
%sql
-- Mise à jour erronée : primes de base de Paris doublées
UPDATE alpenia_dev.bronze.brz_pricing
SET base_premium_eur = base_premium_eur * 2
WHERE city = 'Paris';

In [0]:
# Après
print("VERSION ACTUELLE (erronée) :")
display(spark.sql(f"SELECT * FROM {PRICING} ORDER BY city, product"))

print(f"VERSION AS OF {v_before} (correcte) :")
display(spark.sql(f"SELECT * FROM {PRICING} VERSION AS OF {v_before} ORDER BY city, product"))

In [0]:
# RESTORE
spark.sql(f"RESTORE TABLE {PRICING} TO VERSION AS OF {v_before}")

# Preuve : la table est identique à la bonne version, dans les deux sens
now = spark.table(PRICING)
old = spark.sql(f"SELECT * FROM {PRICING} VERSION AS OF {v_before}")
assert now.exceptAll(old).count() == 0 and old.exceptAll(now).count() == 0
print("RESTORE OK : contenu identique à la version", v_before)

In [0]:
history(PRICING)
# Attendu : une ligne CREATE OR REPLACE TABLE AS SELECT par exécution de 02 et de 03, puis UPDATE, puis RESTORE.
# RESTORE crée une NOUVELLE version : la version de l'UPDATE reste dans l'historique.

# OPTIMIZE sur brz_web_events

In [0]:
# Avant
def detail(table):
    return (spark.sql(f"DESCRIBE DETAIL {table}")
            .select("numFiles", "sizeInBytes").first())

d_before = detail(EVENTS)
print(f"AVANT : {d_before.numFiles} fichiers, {d_before.sizeInBytes:,} octets")

In [0]:
%sql
OPTIMIZE alpenia_dev.bronze.brz_web_events;

In [0]:
d_after = detail(EVENTS)
print(f"APRÈS : {d_after.numFiles} fichiers, {d_after.sizeInBytes:,} octets")

display(
    spark.sql(f"DESCRIBE HISTORY {EVENTS}")
    .filter("operation = 'OPTIMIZE'")
    .select("version", "operation",
            F.col("operationMetrics.numRemovedFiles").alias("files_removed"),
            F.col("operationMetrics.numAddedFiles").alias("files_added"))
)

* OPTIMIZE regroupe de nombreux petits fichiers en quelques gros fichiers. Moins de fichiers, c'est moins de métadonnées à lire et des lectures plus rapides.
* Les anciens fichiers ne sont pas supprimés physiquement : ils sont marqués comme retirés dans le journal Delta, donc le time travel fonctionne toujours. Seul VACUUM les supprime.
* La baisse de numFiles dépend du nombre de petits fichiers laissés par l'écriture. Si la table n'avait déjà qu'un ou deux fichiers, l'écart sera faible. Databricks peut aussi compacter automatiquement les tables managées : un faible écart n'est donc pas une erreur.

# VACUUM

In [0]:
%sql
-- Fichiers qui seraient supprimés (aucun attendu : ils ont tous moins de 7 jours)
VACUUM alpenia_dev.bronze.brz_pricing DRY RUN;

In [0]:
%sql
-- VACUUM réel, avec la rétention par défaut (7 jours)
VACUUM alpenia_dev.bronze.brz_pricing;

In [0]:
# Time travel après VACUUM : vers la version d'avant la mise à jour erronée, puis vers la version erronée elle-même
for v in (v_before, v_before + 1):
    print(f"VERSION AS OF {v} :")
    display(spark.sql(f"SELECT * FROM {PRICING} VERSION AS OF {v} ORDER BY city, product"))

# VACUUM apparaît dans l'historique (VACUUM START / VACUUM END)
history(PRICING)

In [0]:
# Tentative de VACUUM avec une rétention nulle : refusée par le contrôle de sécurité de Delta
try:
    spark.sql(f"VACUUM {PRICING} RETAIN 0 HOURS")
    print("VACUUM RETAIN 0 HOURS exécuté : les versions antérieures ne sont plus lisibles")
except Exception as e:
    print("Refusé :", str(e).splitlines()[0])

**Observation.** Après le VACUUM, le time travel fonctionne toujours, vers la version d'avant la mise à jour erronée comme vers la version erronée elle-même.

* VACUUM ne supprime que les fichiers **qui ne sont plus référencés par la version actuelle** et **retirés depuis plus longtemps que la rétention** (`delta.deletedFileRetentionDuration`, 7 jours par défaut). Ici, tous les fichiers ont été retirés il y a quelques minutes : rien n'est supprimé.
* De plus, `RESTORE` a ré-ajouté à la version actuelle les fichiers de la version d'avant l'UPDATE : ces fichiers sont à nouveau référencés, VACUUM ne les supprimerait jamais tant que la table reste dans cet état.

**Quand le time travel cesserait de fonctionner.** Une version n'est plus lisible dès que l'un de ses fichiers de données a été supprimé, c'est-à-dire après un VACUUM exécuté plus de 7 jours après le retrait de ces fichiers (ou plus tôt si la rétention est abaissée). Ici, la version erronée (`v_before + 1`) deviendrait illisible en premier, puisque ses fichiers ne sont plus référencés. Le journal lui-même est aussi purgé après `delta.logRetentionDuration` (30 jours par défaut).

Abaisser la rétention à 0 heure est refusé par défaut (cellule précédente) : c'est une protection contre la perte de l'historique et contre la suppression de fichiers encore lus par une requête en cours.